In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import DataLoader, Subset

import random
import numpy as np
random.seed(666)

anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles.csv'

## Training ?? !!
Pulling this from the `train_step` function in Travis's `training_functions.py`

In [3]:
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import ContraSeqDataset
from losses import SupConLoss

import torch
# from tqdm.auto import trange, tqdm

torch.cuda.empty_cache()

use_cuda = True
device =  torch.device("cuda" if use_cuda else "cpu")

## Dataloader
def collate_fn(samp):
    # custom collate function required bc diff number of augs
    max_len = max([len(x) for x in samp])
    for fuzz_ball in samp:
        while len(fuzz_ball) < max_len:
            dummy = {k:None for k in samp[0][0].keys()}
            fuzz_ball.append(dummy)
    return samp

In [4]:
# from contra_seq_dataset import ContraSeqDataset
# samp = ds[:6]
# samp_dum = collate_fn(samp)
# print(len(samp_dum))
# print(len(samp_dum[0]))

## batch size of 6 !!!

In [18]:
from seqAE_model import SeqAutoencoder
from losses import SupConLoss, padce_loss
from tqdm.notebook import tqdm

bs = 6

supcon_loss = SupConLoss()

## Dataset
ds = ContraSeqDataset(anc_path, aug_path)

## Model instantiation
model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)

lr = 0.00001
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = None 
model = model.to(device)
model = model.train()
    


loader = DataLoader(ds, bs, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_fn)
# data_iter = tqdm(enumerate(loader),total=len(ds))

for samp in tqdm(loader,total=(len(ds)//6)):
    
    fuzz_size = len(samp[0])
    labels = np.zeros((fuzz_size)*bs)
    for i in range(bs):
        labels[(i*fuzz_size):(i*fuzz_size)+fuzz_size+1] = i
    samp = sum(samp, [])
    labels = np.delete(labels, [i for i in range(len(samp)) if samp[i]['seq']==None ])
    labels = torch.tensor(labels)
    samp = [x for x in samp if x['seq']!=None]
    
    optimizer.zero_grad()
    
    latent = []
    for fuzz_ball in samp:
        for key, value in fuzz_ball.items():
                fuzz_ball[key] = fuzz_ball[key].to(device)
        latent_vec, dec_out = model.forward(**fuzz_ball)
        latent.append(latent_vec)
    latent = torch.cat(latent)
    latent = latent.unsqueeze(1)

    contra_loss = supcon_loss(latent, labels=labels)
    dec_out = dec_out.squeeze()
    print(dec_out.shape)
    recon_loss = padce_loss(fuzz_ball['seq'],dec_out,fuzz_ball['pad_mask'],
                            fuzz_ball['out_mask'])
    
    print(contra_loss.item(), recon_loss.item())
    loss = contra_loss + recon_loss
    loss.backward()
    optimizer.step()
torch.cuda.empty_cache()

  0%|          | 0/1666 [00:00<?, ?it/s]

torch.Size([122, 31])
torch.Size([122]) torch.Size([122, 31]) torch.Size([1, 122]) torch.Size([1, 31])


IndexError: too many indices for tensor of dimension 1

In [ ]:
#         if fuzz_ball['seq']==None:
#             latent_vecs.append(torch.zeros(1,32))
#             continue

In [ ]:
#     print([x['smiles'] for x in samp])

In [ ]:
SupConLoss(nn.Module):
    def __init__(self, temp=0.07, contrast_mode='all', base_temp=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temp
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temp

    def forward(self, features, labels=None, mask=None):

In [ ]:
    for k in sample.keys():
#         for 
        print(k,v)
        sample[k] = sample[k].to(device)

In [ ]:
    
#         print(dec_out.shape)
#         print(dec_out)
    
#     n_augs = len(samp['augs'][0])
#     mask = np.zeros((n_augs+1,n_augs+1))
#     for idx in range(2):
#         print(anc[idx]['smiles'])
#         print(idx)
#     print(samp)

    
#     print(anc[0]['smiles'])
#     print([augs[0][i]['smiles'] for i in range(len(augs[0]))])
    
#     print(len(anc))
#     print(len(augs), len(augs[0]))
    

In [ ]:
train_ds.shape

In [ ]:
# Fitting
for i,samp in data_iter:
    for k,v in samp.items():
        print(k,v)
        sample[key] = sample[key].to(device)

    # Get loss ...
    # 
    #
    #

 

In [ ]:
    loss = train_step(model, sample, optimizer, 
                      kl_alpha = kl_annealer.weight(i),
                     variational = variational,
                     use_out_mask = use_out_mask,
                     prop_pred_loss = prop_pred_loss,
                     recon_alpha = recon_alpha)